In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

In [2]:
data= r"C:\Users\rajeshkumar.t\Downloads\Agn.csv"
df = pd.read_csv(data).replace("noData",np.nan)
print(df.columns)

Index(['incident_id', 'customer_id', 'order_external_id', 'order_item_id_list',
       'incident_created_by', 'incident_owner', 'parent_issues',
       'final_issue_type', 'final_issue_type_id', 'issue_type',
       'sub_issue_type', 'sub_sub_issue_type', 'status_name', 'status_id',
       'promise_id', 'resolution_deadline', 'resolution_deadline_string',
       'partner_response_deadline', 'partner_resolution_deadline',
       'cs_response_deadline', 'assigned_to_queue_at', 'current_queue_name',
       'current_queue_id', 'assigned_agent', 'assigned_to_agent_at',
       'long_threads', 'subject', 'allocation_changed_at',
       'shipment_in_transit', 'task_disposition_id', 'task_assigned_to_id',
       'task_issue_type_name', 'allocation_status', 'return_reason', 'hub',
       'hub_action', 'process_instance_id', 'disposition_id', 'fbf',
       'should_track_manually', 'courier_partner', 'should_confirm_promise',
       'disposition_name', 'follow_up', 'follow_update',
       'inciden

In [ ]:
le_agent = LabelEncoder()
le_queue = LabelEncoder()
df['assigned_agent_enc'] = le_agent.fit_transform(df['assigned_agent'].astype(str))
df['queue_enc'] = le_queue.fit_transform(df['current_queue_id'].astype(str))

X_mlp = df[['assigned_agent_enc', 'queue_enc', 'pain_in_mins', 'Total_count','incident_ijs_score']]
y_mlp = df['resolution_deadline_breached']

X_s = StandardScaler().fit_transform(X_mlp.fillna(0))
mlp = MLPClassifier(hidden_layer_sizes=(100,50), activation = 'relu', solver='adam').fit(X_s, y_mlp)

perm_importance = permutation_importance(mlp, X_s, y_mlp)
plt.barh(X_mlp.columns, perm_importance.importances_mean, color= 'teal')
plt.title("MLP Drivers")
plt.show()
                                

In [ ]:
#AutoEncoder
from tensorflow.keras import layers, models

X_ae = df[['pain_in_mins', 'ijs_score', 'thread_count', 'is_breached']].fillna(0)
X_ae_s = StandardScaler().fit_transform(X_ae)

input_layer = layers.Input(shape=(4,))
encoded = layers.Dense(2, activation = 'relu') (input_layer)
decoded = layers.Dense(4, activation = 'sigmoid')(encoded)
autoencoder = models.Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_ae_s, X_ae_s, epochs= 30, batch_size =16, verbose=0)

recons = autoencoder.predict(X_ae_s)
errors = np.mean(np.square(X_ae_s - recons), axis=1)

sns.histplot(errors, kde=True, color='orange')
plt.title("Autoencoder")
plt.show()     
          

In [ ]:
#Convolutional Neural Network

X_cnn = df[['issue_type_id', 'status_id', 'current_queue_id', 'first_queue_id','thread_count']]
X_cnn = X_cnn.values.reshape(X_cnn.shape[0], X_cnn.shape[1],1)
y_cnn = df['is_escalated'].fillna(0)

model_cnn = models.Sequential([
    layers.Conv1D(32, kernel_size=2, activation='relu', imput_shape=(4,1)),
    layers.GlobalMaxPooling1D(),
    layers.Dense(1, activation='sigmoid')
])
model_cnn.compile(optimizer='adam', loss= 'binary_crossentropy', metrics=['accuracy'])
history = model_cnn.fit(X_cnn, y_cnn, epochs=20, verbose=0)

plt.plot(history.history['loss'], label='Training Loss')
plt.title("CNN")
plt.legend()
plt.show()



In [ ]:
#RNN / LSTM

X_rnn = df[[ 'pain_in_mins', 'is_escalated', 'is_blacklisted']].fillna(0).values
X_rnn = X_rnn.reshape(X_rnn.shape[0], 1,3)
y_rnn = df['breach_ageing'].fillna(0)
#y_rnn = df['resolution_deadline_breached'].map({'Yes': 1, 'No': 0}).fillna(0)

model_rnn = models.Sequential([
    layers.LSTM(32, input_shape=(1,3)),
    layers.Dense(1, activation= 'sigmoid')
])

model_rnn.compile(optimizer='adam' , loss ='binary_crossentropy',metrics=['accuracy'])
model_rnn.fit(X_rnn, y_rnn, epochs=20, verbose=0)

from sklearn.metrics import confusion_matrix
sns.heatmap(confusion_matrix(y_rnn, (model_rnn.predict(X_rnn) > 0.5)), annot= True, cmap='Blues')
plt.title("RNN Temperoal Breach")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [ ]:
def create_sequences(data, seq_length):
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_length):
        X_seq.append(data[i:(i + seq_length)])
        y_seq.append(data[i + seq_length])
        return np.array(X_seq), np.array(y_seq)

daily_data = df.groupby('partition_date')['is_breached'].sum().values.reshape(-1, 1)

if len(daily_data) > 7 :
    X_ts, y_ts = create_sequences(daily_data,7)
    X_ts =X.ts.reshape(X_ts.shape[0], 7,1)

    model_forecast = Sequential([
    LSTM(50, activation='relu', imput_shape=(7,1)),
    Dense(1)
    ])

model_forecast.compile(optimizer = 'adam', loss='mse')
model_forecast.fit(X_ts, y_ts, epochs=50, verbose=0)

last_7_days = daily_data[-7:].reshape(1,7,1)
tomorrow_forecast = model_forecast.predict(last_7_days)

print(f"Predicted Breaches for timmorrow: {int(tomorrow_forecast[0][0])}")